<a href="https://colab.research.google.com/github/fvarellalopes/clawsouls/blob/main/Z_Image_Turbo_4bit_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-yhy6u3si
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-yhy6u3si
  Resolved https://github.com/huggingface/diffusers to commit 48f39c2d59e8db444cb37f91e72413a1db9a2dd6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/Disty0/sdnq to /tmp/pip-req-build-ff07o9w3
  Running command git clone --filter=blob:none --quiet https://github.com/Disty0/sdnq /tmp/pip-req-build-ff07o9w3
  Resolved https://github.com/Disty0/sdnq to commit db8fad818997c259c973a04b6e6963aa8561c170
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
import diffusers
from sdnq import SDNQConfig # import sdnq to register it into diffusers and transformers
from sdnq.loader import apply_sdnq_options_to_model

pipe = diffusers.ZImagePipeline.from_pretrained("Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32", torch_dtype=torch.float32, device_map="cuda")
pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1406 [00:00<?, ?it/s]

In [3]:
# prompt = "a grizzled 60-year-old mage, his face a striking fusion of ancient mysticism and cutting-edge cyberware, staring directly into the camera with piercing, bioluminescent eyes that flicker between arcane violet and cold machine blue. His silver-streaked beard is woven with delicate gold circuitry, pulsing faintly with energy, while the left side of his face transitions seamlessly into sleek, blackened metal plating, etched with glowing runes that hum with latent power. His robe, a tattered mix of enchanted fabric and nano-weave armor, clings to his broad shoulders, its frayed edges crackling with unstable magic. Behind him, a sprawling microchip cityscape throbs with neon-yellow circuit patterns against an abyssal black void, the labyrinthine pathways mirroring the intricate scars and implants across his weathered skin. The air around him shimmers with distortion—part holographic spell matrix, part overheating processor—as if reality itself struggles to contain him."
# image = pipe(
#     prompt=prompt,
#     height=1024,
#     width=1024,
#     num_inference_steps=9,
#     guidance_scale=0.0,
#     generator=torch.manual_seed(42),
# ).images[0]
# display(image)

In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [5]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from io import BytesIO
import base64
import torch

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"])

TOKEN = 'cs-secret-2026'

class Req(BaseModel):
    prompt: str

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: Req, x_token: str = None):
    if x_token != TOKEN:
        raise HTTPException(status_code=401, detail="Invalid")
    image = pipe(prompt=req.prompt, height=1024, width=1024,
                 num_inference_steps=9, guidance_scale=0.0,
                 generator=torch.manual_seed(42)).images[0]
    buf = BytesIO()
    image.save(buf, format="PNG")
    return {"image": base64.b64encode(buf.getvalue()).decode()}


In [12]:
import subprocess
import re, time

subprocess.Popen(['python', '-m', 'uvicorn', '__main__:app', '--host', '0.0.0.0', '--port', '8081'])
time.sleep(10)

In [6]:

import subprocess
import re, time

CLOUDFLARE_TOKEN = "eyJhIjoiNmIxNmYwMzUwNjM5NWFhNjBjZjk2NzY0MDA2Y2I0MGUiLCJ0IjoiYzA1MzQ2NGEtNzBhYy00MmUwLWFkMjQtZDdiMDFkYzBmOGZhIiwicyI6Ik1qbGtOekEyWVRjdFkyWXdOQzAwTXpGa0xXSTNZV1V0WkdJek5HVTJNRGhrT0RVeCJ9"

cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TOKEN, '--url', 'http://localhost:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Espera a URL do tunnel aparecer nos logs
tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('🌐 TÚNEL ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print()
    print('Endpoints:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
    print()
    print('📋 Copie esta URL e cole aqui no chat para eu usar!')
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cloudflared_proc.wait()')



⏳ Esperando URL do tunnel...
2026-05-10T01:25:08Z INF Starting tunnel tunnelID=c053464a-70ac-42e0-ad24-d7b01dc0f8fa
2026-05-10T01:25:08Z INF Version 2026.3.0 (Checksum 4a9e50e6d6d798e90fcd01933151a90bf7edd99a0a55c28ad18f2e16263a5c30)
2026-05-10T01:25:08Z INF GOOS: linux, GOVersion: go1.24.13, GoArch: amd64
2026-05-10T01:25:08Z INF Settings: map[token:***** url:http://localhost:8080]
2026-05-10T01:25:08Z INF Autoupdate frequency is set autoupdateFreq=86400000
2026-05-10T01:25:08Z INF Generated Connector ID: f51c7a51-3709-4197-b2aa-a79eca54a1e5
2026-05-10T01:25:08Z INF Initial protocol quic
2026-05-10T01:25:08Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-10T01:25:08Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-10T01:25:08Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-10T01:25:08Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-10T01:25:08Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-05-10T01:25:0

KeyboardInterrupt: 

In [13]:
import subprocess

subprocess.run(['ps', 'aux'], text=True, capture_output=True)

CompletedProcess(args=['ps', 'aux'], returncode=0, stdout='USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND\nroot           1  0.0  0.0    988   564 ?        Ss   00:23   0:00 /sbin/docker-init -- /datalab/run.sh\nroot           7  0.3  0.5 1302548 66972 ?       Sl   00:23   0:14 /tools/node/bin/node /datalab/web/app.js\nroot           9  0.0  0.0   7376  3400 ?        S    00:23   0:02 /bin/bash -e /usr/local/colab/bin/oom_monitor.sh\nroot          11  0.0  0.0   7376  1928 ?        S    00:23   0:00 /bin/bash -e /datalab/run.sh\nroot          12  0.8  0.1 1272280 20260 ?       Sl   00:23   0:32 /usr/colab/bin/kernel_manager_proxy --listen_port=6000 --target_port=9000 --logtostderr --listen_host=172.28.0.12 --target_host=172.28.0.12 --tunnel_background_save_url=https://colab.research.google.com/tun/m/cc48301118ce562b961b3c22d803539adc1e0c19/gpu-t4-s-kkb-ass1c2-1bi01clwzl021 --tunnel_background_save_delay=10s --tunnel_periodic_background_save_frequency=30m0s -